# 06 — Association Rule Generation & Evaluation

**Objective:** Generate and evaluate association rules from the frequent itemsets mined by Apriori and FP-Growth using `mlxtend.frequent_patterns.association_rules`.

### Implementation Checklist:
1. **Take frequent itemsets** generated by Apriori (`min_support = 0.01`).
2. **Generate association rules** using `association_rules()`.
3. **Calculate key metrics**: Support, Confidence, Lift, Leverage, and Conviction.
4. **Apply minimum confidence** threshold.
5. **Filter meaningful rules** (pruning rules with Lift ≤ 1).
6. **Sort rules based on Lift** and Confidence.
7. **Save the results** in CSV format.
8. **Repeat the process** using FP-Growth itemsets (including deeper rules at `min_support = 0.005`).
9. **Derive business actions** for product placement, cross-selling, and bundling.

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

sns.set_style('whitegrid')

---
## 2. Load Frequent Itemsets

In [ ]:
# Load cleaned transaction data & encode
df = pd.read_csv('../dataset/processed/cleaned_data.csv')
df['Description'] = df['Description'].str.strip()
transactions = df.groupby('InvoiceNo')['Description'].apply(list).tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# Generate itemsets with min_support = 0.01
apriori_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)
fpgrowth_itemsets = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)

print(f"Apriori frequent itemsets (min_sup=0.01)   : {len(apriori_itemsets)}")
print(f"FP-Growth frequent itemsets (min_sup=0.01) : {len(fpgrowth_itemsets)}")

---
## 3. Association Rule Mining on Apriori Itemsets

### Mathematical Definitions of Key Metrics:
- **Support**: $P(A \cup B)$ — proportion of transactions containing both $A$ and $B$.
- **Confidence**: $P(B|A) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$ — probability of purchasing $B$ given that $A$ is purchased.
- **Lift**: $\frac{P(B|A)}{P(B)} = \frac{\text{Support}(A \cup B)}{\text{Support}(A) \times \text{Support}(B)}$ — ratio of observed joint occurrence to expected independent occurrence.
  - $\text{Lift} > 1$: Positive association (items complement each other).
  - $\text{Lift} = 1$: Independence (no correlation).
  - $\text{Lift} < 1$: Negative association (substitutes).

In [ ]:
# Generate rules with minimum confidence = 0.2 (20%)
rules_apriori = association_rules(apriori_itemsets, metric="confidence", min_threshold=0.2)

print(f"Initial Apriori rules generated (min_conf=0.2) : {len(rules_apriori)}")

# Filter meaningful rules (Lift > 1.2)
rules_apriori_filtered = rules_apriori[rules_apriori['lift'] > 1.2].copy()
rules_apriori_filtered = rules_apriori_filtered.sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f"Meaningful rules after filtering (Lift > 1.2) : {len(rules_apriori_filtered)}")

In [ ]:
# Format antecedents and consequents for clean display
display_rules = rules_apriori_filtered.copy()
display_rules['antecedents'] = display_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
display_rules['consequents'] = display_rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Display Top 10 Apriori Rules by Lift
display_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']].head(10)

---
## 4. Repeat Process Using FP-Growth Itemsets

We repeat rule generation on FP-Growth itemsets and verify that the discovered rules are identical.

In [ ]:
rules_fpgrowth = association_rules(fpgrowth_itemsets, metric="confidence", min_threshold=0.2)
rules_fpgrowth_filtered = rules_fpgrowth[rules_fpgrowth['lift'] > 1.2].sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f"FP-Growth meaningful rules generated: {len(rules_fpgrowth_filtered)}")
print(f"Are rule counts identical between Apriori and FP-Growth? → {len(rules_apriori_filtered) == len(rules_fpgrowth_filtered)}")

### Mining Deeper Multi-Item Rules via FP-Growth (`min_support = 0.005`)
Because FP-Growth can handle lower support thresholds without memory exhaustion, we extract higher-order bundles.

In [ ]:
fpgrowth_005 = fpgrowth(df_encoded, min_support=0.005, use_colnames=True)
rules_fp_005 = association_rules(fpgrowth_005, metric="confidence", min_threshold=0.3)
rules_fp_005_filtered = rules_fp_005[rules_fp_005['lift'] > 1.2].sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f"Deep FP-Growth rules at min_support=0.005 & min_confidence=0.3: {len(rules_fp_005_filtered):,}")

In [ ]:
# Top 5 deep rules
display_005 = rules_fp_005_filtered.copy()
display_005['antecedents'] = display_005['antecedents'].apply(lambda x: ', '.join(list(x)))
display_005['consequents'] = display_005['consequents'].apply(lambda x: ', '.join(list(x)))
display_005[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5)

---
## 5. Visualizing Association Rules

In [ ]:
# Scatter plot: Support vs Confidence colored by Lift
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    rules_apriori_filtered['support'] * 100, 
    rules_apriori_filtered['confidence'] * 100, 
    c=rules_apriori_filtered['lift'], 
    cmap='viridis', 
    s=60, 
    alpha=0.8, 
    edgecolors='w'
)
cbar = plt.colorbar(scatter)
cbar.set_label('Lift', fontsize=12, fontweight='bold')

plt.title('Association Rules: Support vs. Confidence (Colored by Lift)', fontsize=14, fontweight='bold')
plt.xlabel('Support (% of Total Transactions)', fontsize=12)
plt.ylabel('Confidence (%)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/graphs/rules_support_vs_confidence_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../results/graphs/rules_support_vs_confidence_scatter.png")

In [ ]:
# Bar chart: Top 10 rules by Lift
top10 = display_rules.head(10).copy()
top10['rule_label'] = [f"{a}  →  {c}" for a, c in zip(top10['antecedents'], top10['consequents'])]
top10_sorted = top10.sort_values(by='lift', ascending=True)

plt.figure(figsize=(12, 7))
ax = plt.barh(top10_sorted['rule_label'], top10_sorted['lift'], color='#673AB7')
plt.title('Top 10 Association Rules Ranked by Lift (min_sup=0.01)', fontsize=14, fontweight='bold')
plt.xlabel('Lift Metric', fontsize=12, fontweight='bold')
plt.ylabel('Association Rule (Antecedent → Consequent)', fontsize=12)

for bar in ax:
    width = bar.get_width()
    plt.text(width + 0.2, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/graphs/top_10_rules_by_lift.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../results/graphs/top_10_rules_by_lift.png")

---
## 6. Save Association Rules to CSV

In [ ]:
# Save Apriori rules
display_rules.to_csv('../results/association_rules/apriori_association_rules_min_sup_0_01.csv', index=False)

# Save FP-Growth rules
rules_fp_export = rules_fpgrowth_filtered.copy()
rules_fp_export['antecedents'] = rules_fp_export['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_fp_export['consequents'] = rules_fp_export['consequents'].apply(lambda x: ', '.join(list(x)))
rules_fp_export.to_csv('../results/association_rules/fpgrowth_association_rules_min_sup_0_01.csv', index=False)

# Save Deep FP-Growth rules (0.005)
display_005.to_csv('../results/association_rules/fpgrowth_association_rules_min_sup_0_005.csv', index=False)

print("All association rules successfully saved to ../results/association_rules/")

---
## 7. Business Insights & Practical Recommendations

### Key Rule Insights:
1. **Regency Teacup Set Bundle (Lift: 21.12, Confidence: 78.5%)**:
   - **Rule**: `{Roses Regency Teacup, Pink Regency Teacup}` $\rightarrow$ `{Green Regency Teacup}`
   - **Interpretation**: When customers purchase both the Roses and Pink teacups, they complete the three-piece collection with the Green teacup **78.5% of the time** (a **21.1× higher likelihood** than random chance).
   - **Action**: Create an official **"Complete Regency Trio" bundle discount** on the product page.

2. **Scandinavian Wooden Ornaments (Lift: 20.01, Confidence: 47.0%)**:
   - **Rule**: `{Wooden Star Scandinavian}` $\rightarrow$ `{Wooden Heart Scandinavian}`
   - **Interpretation**: Strong theme-matching decorative purchase.
   - **Action**: Place matching wooden decorations side-by-side both on web category pages and physical retail shelves.

3. **Lunch Bag Cross-Selling (Lift: ~8-12, Confidence: ~30-40%)**:
   - **Rule**: `{Lunch Bag Red Retrospot}` $\rightarrow$ `{Lunch Bag Black Skull}` / `{Lunch Bag Cars Blue}`
   - **Interpretation**: Customers often buy multiple lunch bags with different designs for multiple family members.
   - **Action**: Offer a **"Buy 2 Lunch Bags, Get 15% Off"** multi-buy promotion.

4. **Hand Warmer Twin Packs (Lift: 16.19, Confidence: 44.2%)**:
   - **Rule**: `{Hand Warmer Scotty Dog}` $\rightarrow$ `{Hand Warmer Owl Design}`
   - **Interpretation**: Winter seasonal gifts frequently bought in pairs.
   - **Action**: Bundle seasonal winter hand warmers together in gift boxes during Q4 holiday marketing.